# QFF 2026 Phase 2 — PS-O1 Weighted Max-Cut

This notebook documents and visualizes the generated benchmark results. Run it from the project root after `solve_ps_o1_phase2.py` has generated `phase2_results.json` and the CSV files. It does not modify the official `qff_phase2.py` or `instances.json`.

**Important:** these are simulator results, not hardware measurements. Confirm all instance and configuration files against the official challenge kit before submission.

In [ ]:
from pathlib import Path
import json, csv
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd()
RESULTS_JSON = ROOT / 'phase2_results.json'
RESULTS_CSV = ROOT / 'results' / 'tables' / 'results.csv'
BELL_CSV = ROOT / 'results' / 'tables' / 'bell.csv'
FIG_DIR = ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
assert RESULTS_JSON.exists(), f'Missing {RESULTS_JSON}; run solve_ps_o1_phase2.py first.'
assert RESULTS_CSV.exists(), f'Missing {RESULTS_CSV}; run solve_ps_o1_phase2.py first.'
data = json.loads(RESULTS_JSON.read_text(encoding='utf-8'))
rows = list(csv.DictReader(RESULTS_CSV.open(encoding='utf-8', newline='')))
bell_rows = list(csv.DictReader(BELL_CSV.open(encoding='utf-8', newline=''))) if BELL_CSV.exists() else []
print('Loaded JSON keys:', list(data.keys()))
print('Benchmark rows:', len(rows), '| Bell rows:', len(bell_rows))

## 1. Processor benchmark summary
The chart uses the mean sampled cut from `results.csv`; Processor A is excluded because it cannot execute a six-qubit PS-O1 circuit.

In [ ]:
usable = [r for r in rows if r.get('executable','').lower() == 'true' and r.get('mean_cut')]
labels = [f"{r['processor']}\n{r['condition']}" for r in usable]
means = [float(r['mean_cut']) for r in usable]
plt.figure(figsize=(8, 4.5))
plt.bar(labels, means)
plt.axhline(13.5, linestyle='--', label='Random-partition expected cut = 13.5')
plt.axhline(22, linestyle=':', label='Exact classical optimum = 22')
plt.ylabel('Mean sampled cut weight')
plt.title('PS-O1 QAOA benchmark — simulator results')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'mean_cut_comparison.png', dpi=180)
plt.show()

## 2. Processor D topology
The diagram below is generated from the current Processor D JSON file. Check it against the final frozen configuration before submission.

In [ ]:
d_path = ROOT / 'processors' / 'processor_D.json'
if d_path.exists():
    d = json.loads(d_path.read_text(encoding='utf-8'))
    G = nx.Graph()
    G.add_nodes_from(range(int(d['num_qubits'])))
    G.add_edges_from([tuple(e) for e in d['edges']])
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(7, 5))
    nx.draw_networkx(G, pos, with_labels=True, node_size=900, font_size=11)
    plt.title('Processor D connectivity')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'processor_D_topology.png', dpi=180)
    plt.show()
else:
    print('processors/processor_D.json not found; topology plot skipped.')

## 3. Bell-state fidelity on Processor D
Fidelity values below come from the Bell CSV. Ideal values should be 1.0 in the simulator; noisy values include the configured noise model.

In [ ]:
if bell_rows:
    chosen = [r for r in bell_rows if r.get('condition') == 'noisy']
    labels = [f"{r['pair_type']}\n({r['q0']},{r['q1']})" for r in chosen]
    vals = [float(r['fidelity']) for r in chosen]
    errs = [float(r['fidelity_se']) for r in chosen]
    plt.figure(figsize=(7, 4.5))
    plt.bar(labels, vals, yerr=errs, capsize=4)
    plt.ylim(0, 1.05)
    plt.ylabel('Bell-state fidelity')
    plt.title('Processor D Bell-state fidelity (noisy simulation)')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'bell_fidelity_noisy.png', dpi=180)
    plt.show()
else:
    print('Bell CSV not found or empty; Bell plot skipped.')

## 4. Sanity checks
The weighted graph has total edge weight 27. A uniformly random bipartition cuts each edge with probability 1/2, so the expected random cut weight is 13.5. With exact optimum 22, the expected random approximation ratio is 13.5 / 22 ≈ 0.613636. Check the CSV field definition in the official kit before assigning a value to `random_baseline`; do not silently mix a cut weight and a ratio.

In [ ]:
total_weight = sum(int(e[2]) for e in json.loads((ROOT / 'data' / 'instance.json').read_text(encoding='utf-8'))['weighted_edges'])
optimum = 22
random_expected_cut = total_weight / 2
print('Total edge weight:', total_weight)
print('Random expected cut:', random_expected_cut)
print('Exact optimum:', optimum)
print('Random expected approximation ratio:', random_expected_cut / optimum)
assert total_weight == 27 and random_expected_cut == 13.5

## Reproducibility notes
- Run from the project root.
- Run `python solve_ps_o1_phase2.py` before this notebook.
- Do not edit the official helper file.
- Save the final notebook after running all cells so outputs are visible.
- Report the Python version and exact package versions used.
- Verify all paths, CSV schema, processor topology, and official requirements before submission.